# 『校則問題』を再考する
## —偏差値と校則の自由度に関する定量的分析—

大森一輝 / 一橋大学ソーシャル・データサイエンス学部２年 / 5125006H

---

### このノートブックについて
データハンドリングの期末レポートの補足資料として、実行コードをこのノートブックに記す。  
データ収集・前処理・形態素解析・アノテーション・スコア計算・統計分析・可視化の全工程を対象とする。

---

## 0. 環境設定

In [23]:
!pip install requests
!pip install beautifulsoup4
!pip install unidic-lite
!pip install mecab-python3
!pip install japanize-matplotlib
!pip install statsmodels -q

In [24]:
import requests
from bs4 import BeautifulSoup
import time
import re

import numpy as np
import pandas as pd
import MeCab

import matplotlib.pyplot as plt
import matplotlib.style
import seaborn as sns
import japanize_matplotlib

from scipy.stats import pearsonr
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

matplotlib.style.use('ggplot')

---

## 1. 校則テキストの収集

### 1-1. HTMLの確認

Requestsを使用して、全国校則一覧（kousoku.org）のwebページ情報を取得する。

In [25]:
r = requests.get('https://www.kousoku.org/')
text = r.text
print(text[:2000])

<!doctype html>
<html lang="ja">

<head>
<meta charset="utf-8">
<meta http-equiv="X-UA-Compatible" content="IE=edge">
<meta name="viewport" content="width=device-width, initial-scale=1.0, viewport-fit=cover"/>
<meta name="referrer" content="no-referrer-when-downgrade"/>


  
    <!-- Global site tag (gtag.js) - Google Analytics -->
    <script async src="https://www.googletagmanager.com/gtag/js?id=G-HMPK2PSN8W"></script>
    <script>
      window.dataLayer = window.dataLayer || [];
      function gtag(){dataLayer.push(arguments);}
      gtag('js', new Date());

      gtag('config', 'G-HMPK2PSN8W');
    </script>

  


  
  

  <!-- Google Search Console -->
<meta name="google-site-verification" content="WfOo69WLdavAI3rE3URJYW8C6XprB7gBJaMlPYz5Kh4" />
<!-- /Google Search Console -->
<!-- preconnect dns-prefetch -->
<link rel="preconnect dns-prefetch" href="//www.googletagmanager.com">
<link rel="preconnect dns-prefetch" href="//www.google-analytics.com">
<link rel="preconnect dns-prefet

上記から、kousoku.orgは校則を都道府県別に `https://www.kousoku.org/{都道府県}/` というディレクトリで管理していることがわかる。

実際に https://www.kousoku.org/tokyo/ を確認してみると、東京都の各公立高校の校則一覧が表示されており、そこから各高校の校則ページへと飛ぶことができる。つまり、kousoku.orgは都道府県ページの下に各学校の校則ページが設置されている構造になっている。

このwebスクレイピングの流れは、まず都道府県ページにアクセスして全国の学校の校則URLの一覧を取得し、そのURL一覧を用いてそれぞれの校則テキストデータを取得していく、というものである。

### 1-2. スクレイピング実験

全体のスクレイピングを始める前に、試しに東京都の校則URL一覧（上から２０件）を取得する。

In [26]:
r_check = requests.get('https://www.kousoku.org/tokyo')
soup_check = BeautifulSoup(r_check.text, 'html.parser')

for a in soup_check.find_all('a')[:20]:
    href = a.get('href')
    if href:
        print('', href)

 https://www.kousoku.org/
 https://www.kousoku.org/tokyo/d113299901013/
 https://www.kousoku.org/tokyo/d213299900059/
 https://www.kousoku.org/tokyo/d213299900040/
 https://www.kousoku.org/tokyo/d213299900013/
 https://www.kousoku.org/tokyo/d213299900022/
 https://www.kousoku.org/tokyo/d213299900031/
 https://www.kousoku.org/tokyo/d113299913019/
 https://www.kousoku.org/tokyo/d113299905126/
 https://www.kousoku.org/tokyo/d113299906198/
 https://www.kousoku.org/tokyo/d113299909168/
 https://www.kousoku.org/tokyo/d113299908169/
 https://www.kousoku.org/tokyo/d113299911011/
 https://www.kousoku.org/tokyo/d113299911048/
 https://www.kousoku.org/tokyo/d113299911020/
 https://www.kousoku.org/tokyo/d113299911039/
 https://www.kousoku.org/tokyo/d113299912010/
 https://www.kousoku.org/tokyo/d113299914018/
 https://www.kousoku.org/tokyo/d113299908123/
 https://www.kousoku.org/tokyo/d113299909060/


次に、各学校の校則ページのHTMLの中身がどうなっているか確認するため、試しに一橋高等学校のHTMLを読み込む。

ウェブサイトを見ると、見出しに「【東京】一橋高等学校（定時・通信）の校則」と書かれていることが読み取れる。そこで、当該高校の名前はh1タグに入っていると考えられる。また、校則本文についても、メインコンテンツであることからarticleタグもしくはdiv class="entry-content"タグに入っていることが予想できる。

In [27]:
r_sample = requests.get('https://www.kousoku.org/tokyo/d113299901013/')
soup_sample = BeautifulSoup(r_sample.text, 'html.parser')

print('h1タグ:', soup_sample.find('h1').text)

content = (
    soup_sample.find('article') or
    soup_sample.find('div', class_='entry-content')
)
if content:
    print(content.text[:500])

h1タグ: 【東京】一橋高等学校（定時・通信）の校則


【東京】一橋高等学校（定時・通信）の校則


 













この学校は公式サイトで校則を公開しています



学校運営方針 ｜ 東京都立一橋高等学校　定時制 | 東京都立学校
学校運営方針 ｜ 東京都立一橋高等学校　通信制 | 東京都立学校





2021.12.162025.02.24

2021年度の校則を掲載していますこのページに掲載している校則は2021年度時点のものです。情報が古くなっている可能性が特にございます。


東京都に対する情報公開請求で開示された2021年度の校則等について掲載しています。
このページの目次

定時制生徒心得登下校時所持品学校生活授業施設利用SNS等インターネット利用の注意禁止事項通信制生徒心得学校生活について


定時制
生徒心得
本校では皆がお互いに気持ちよく、そして有意義な学校生活を送れるよう、以下のように生徒心得を定めている。いずれも学校や社会で生活していく上での必要最低限のマナーなので、授業や部活動、課外活動の時間や登下校時をはじめとして、あらゆる場所でしっかり心に留め置いて行動すること。
登下校時

下校の時間を


実際、h1タグに学校名、div class="entry-content"タグに校則本文が格納されていることが確認できた。次に、一橋高等学校の校則データを取得する。

In [28]:
r_hit = requests.get('https://www.kousoku.org/tokyo/d113299901013/')
soup_hit = BeautifulSoup(r_hit.text, 'html.parser')

print('h1タグ:', soup_hit.find('h1').text)

content = (
    soup_hit.find('article') or
    soup_hit.find('div', class_='entry-content')
)
if content:
    print(content.text[:300])

h1タグ: 【東京】一橋高等学校（定時・通信）の校則


【東京】一橋高等学校（定時・通信）の校則


 













この学校は公式サイトで校則を公開しています



学校運営方針 ｜ 東京都立一橋高等学校　定時制 | 東京都立学校
学校運営方針 ｜ 東京都立一橋高等学校　通信制 | 東京都立学校





2021.12.162025.02.24

2021年度の校則を掲載していますこのページに掲載している校則は2021年度時点のものです。情報が古くなっている可能性が特にございます。


東京都に対する情報公開請求で開示された2021年度の校則等について掲載しています。
このページの目次

定時制生徒心得登下校時所持品学校生活授


### 1-3. 校則データの取得

都道府県ページにアクセスし、全国の学校の校則URLの一覧を取得し、そのURL一覧を用いてそれぞれの校則テキストデータを取得していく。まずは、全国の校則URLをリスト化する関数を作成する。

In [29]:
def school_urls(pref, delay=1.5):
    url = 'https://www.kousoku.org/' + pref + '/'
    r = requests.get(url, timeout=10)
    soup = BeautifulSoup(r.text, 'html.parser')

    url_list = list()
    for a in soup.find_all('a'):
        href = a.get('href', '')
        # 都道府県ページ直下の学校ページURLのみ取得する
        # 形式：/tokyo/d113.../ のようなパスになっている
        if href.startswith('https://www.kousoku.org/' + pref + '/d'):
            if href not in url_list:
                url_list.append(href)

    time.sleep(delay)
    return url_list

In [30]:
# 動作確認
urls_test = school_urls('tokyo')
for u in urls_test[:3]:
    print('', u)

 https://www.kousoku.org/tokyo/d113299901013/
 https://www.kousoku.org/tokyo/d213299900059/
 https://www.kousoku.org/tokyo/d213299900040/


次に、校則URL関数を利用して、学校ページから学校名・都道府県・年度・本文を取得する校則テキスト関数を作成する。学校名はh1タグに「【〇〇県】◯◯高等学校の校則」と書いてあるため、【〇〇県】や「の校則」は除去する。

In [31]:
def scrape_school(url, delay=2.0):
    try:
        r = requests.get(url, timeout=15)
        soup = BeautifulSoup(r.text, 'html.parser')

        # 学校名の抽出
        h1 = soup.find('h1')
        title = h1.text.strip() if h1 else ''
        if '】' in title:
            school_name = title.split('】')[1]
        else:
            school_name = title
        # 「の校則」を除去する
        if 'の校則' in school_name:
            school_name = school_name.replace('の校則', '')
        school_name = school_name.strip()

        # 都道府県の抽出（URLのパスから取得する）
        pref = url.split('kousoku.org/')[1].split('/')[0]

        # 年度の抽出（ページ内のテキストから探す）
        year_tag = soup.find(text=re.compile(r'20\d\d年度'))
        year = year_tag.strip()[:4] if year_tag else None

        # 校則本文の抽出
        content = (
            soup.find('article') or
            soup.find('div', class_='entry-content')
        )
        raw_text = content.text if content else ''

        time.sleep(delay)
        return {
            'school_name': school_name,
            'prefecture':  pref,
            'year':        year,
            'url':         url,
            'raw_text':    raw_text,
        }
    except Exception as e:
        print('  エラー：', url, str(e))
        time.sleep(delay)
        return None

In [32]:
# 動作確認
scrape_test = scrape_school(urls_test[0])
print('学校名：',          scrape_test['school_name'])
print('都道府県：',        scrape_test['prefecture'])
print('年度：',            scrape_test['year'])
print('本文（先頭200文字）：\n' + scrape_test['raw_text'][:200])

/var/folders/9v/t_x78pcj1x5c11t91l365mtw0000gn/T/ipykernel_16963/2542545621.py:22: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  year_tag = soup.find(text=re.compile(r'20\d\d年度'))


学校名： 一橋高等学校（定時・通信）
都道府県： tokyo
年度： {"@c
本文（先頭200文字）：


【東京】一橋高等学校（定時・通信）の校則


 













この学校は公式サイトで校則を公開しています



学校運営方針 ｜ 東京都立一橋高等学校　定時制 | 東京都立学校
学校運営方針 ｜ 東京都立一橋高等学校　通信制 | 東京都立学校





2021.12.162025.02.24

2021年度の校則を掲載していますこのページに掲載している校則は2021年度時点のも


動作確認ができたので、全国全都道府県の校則データを取得する。kousoku.orgに掲載がある都道府県のみを対象とする。既にCSVが保存済みの場合は `SKIP_SCRAPING = True` に変更することで読み込みをスキップできる。

In [33]:
PREF_LIST = [
    'hokkaido',
    'aomori', 'iwate', 'miyagi', 'akita', 'yamagata', 'fukushima',
    'ibaraki', 'tochigi', 'gunma', 'saitama', 'chiba', 'tokyo', 'kanagawa',
    'niigata', 'toyama', 'ishikawa', 'fukui', 'yamanashi', 'nagano',
    'shizuoka', 'aichi', 'gifu', 'mie',
    'shiga', 'kyoto', 'osaka', 'hyogo', 'nara', 'wakayama',
    'tottori', 'shimane', 'okayama', 'hiroshima', 'yamaguchi',
    'tokushima', 'kagawa', 'ehime',
    'fukuoka', 'saga', 'nagasaki', 'kumamoto', 'oita', 'miyazaki',
    'kagoshima', 'okinawa',
]

# 既にschool_rules.csvが保存済みの場合はTrueに変更する
SKIP_SCRAPING = True

if SKIP_SCRAPING:
    df_schools = pd.read_csv('school_rules.csv')
    print(len(df_schools), '校の校則を読み込んだ。')

else:
    all_records = list()
    for pref in PREF_LIST:
        urls = school_urls(pref)
        if len(urls) == 0:
            print(pref, '：掲載なし')
            continue
        print(pref, '：', len(urls), '校')
        for url in urls:
            record = scrape_school(url)
            if record:
                all_records.append(record)
        # 20校ごとに途中保存する
        if len(all_records) % 20 == 0:
            pd.DataFrame(all_records).to_csv('school_rules.csv', index=False, encoding='utf-8-sig')

    df_schools = pd.DataFrame(all_records)
    df_schools.to_csv('school_rules.csv', index=False, encoding='utf-8-sig')
    print('収集完了：', len(df_schools), '校')

1666 校の校則を読み込んだ。


---

## 2. 偏差値データの収集

偏差値データは「高校偏差値.net」から取得する。

当初は先行研究（大森、2023）でも参照した「みんなの高校情報」の使用を検討していた。しかし、利用規約を確認した結果、スクレイピングが禁止されていることが判明したため、データソースを変更した。「高校偏差値.net」はrobots.txtに実質的な制限がなく、利用規約においても商業利用のみが禁止されていることを確認した。

### 2-1. HTMLの確認

スクレイピング実装前に、都道府県ページのHTMLを確認する。以下、東京都を例に考える。

In [34]:
r_h = requests.get('https://xn--swqwd788bm2jy17d.net/tokyo.php')
text_h = r_h.text
print(text_h[:2000])

﻿﻿
	
<!DOCTYPE HTML PUBLIC '-//W3C//DTD HTML 4.01 Transitional//EN' 'http://www.w3.org/TR/html4/loose.dtd'>
<HTML lang='ja-JP'>
<head>
<meta http-equiv='Content-Type' content='text/html; charset=utf-8' />
<meta http-equiv='content-style-type' content='text/css'>
<meta http-equiv='Content-Script-Type' content='text/javascript'>
	<LINK REL="SHORTCUT ICON" HREF="./images/icon.gif" type="image/gif">
	<LINK REL="ICON" HREF="./images/icon.gif" type="image/gif">
<link rel="apple-touch-icon" href="https://高校偏差値.net/images/apple-touch-icon.png">
<link rel="stylesheet" href="https://高校偏差値.net/base.css" type="text/css">
<script type="text/javascript" src="https://高校偏差値.net/js/jquery-1.10.2.min.js"></script>
<script type="text/javascript" src="https://高校偏差値.net/js/movable.js"></script>
<link rel="stylesheet" href="https://use.fontawesome.com/releases/v6.0.0/css/all.css">

<meta name="description" content="東京都の2026年最新の高校偏差値を学科・コース別にランキング。各高校の偏差値のボーダーラインから受験倍率、高校の詳細情報も紹介していますので希望の高校が見つかります。">
<meta 

上記から、各都道府県ページはHTMLのtableタグで構造化されており、trタグが学校ごとの行に対応している。次に、東京ページの表の先頭数行を確認する。

In [35]:
soup_h = BeautifulSoup(r_h.text, 'html.parser')

table_tokyo = soup_h.find('table')
if table_tokyo:
    print('テーブルの先頭3行：')
    for tr in table_tokyo.find_all('tr')[:10]:
        tds = tr.find_all('td')
        print([td.text.strip() for td in tds])

テーブルの先頭3行：
[]
['78', 'お茶の水女子大学附属高校', '国立', '女子', '普通', '3.26', '144年', '75～81', '1/640位', '文京区', 'S']
['78', '開成高校', '私立', '男子', '普通', '2.94', '155年', '75～81', '1/640位', '荒川区', 'S']
['78', '筑波大学附属駒場高校', '国立', '男子', '普通', '2.93', '76年', '75～81', '1/640位', '世田谷区', 'S']
['78', '筑波大学附属高校', '国立', '共学', '普通', '3.78', '138年', '75～81', '1/640位', '文京区', 'S']
['77', '慶應義塾女子高校', '私立', '女子', '普通', '3.44', '76年', '74～80', '5/640位', '港区', 'S']
['77', '東京学芸大学附属高校', '国立', '共学', '普通', '3.49', '72年', '74～80', '5/640位', '世田谷区', 'S']
['76', '早稲田実業学校高等部', '私立', '共学', '普通', '3.77', '125年', '73～79', '7/640位', '国分寺市', 'S']
['76', '早稲田大学高等学院', '私立', '男子', '普通', '2.73', '106年', '73～79', '7/640位', '練馬区', 'S']
['74', '国立高校', '都立', '共学', '普通', '1.47', '86年', '71～77', '9/640位', '国立市', 'S']


上記から、各列の内容は以下のとおりであることが確認できた。

| インデックス | 内容 |
|---|---|
| 0 | 偏差値 |
| 1 | 学校名 |
| 2 | 設置者（都立・私立・国立等） |
| 3 | 男女別（共学・女子・男子） |
| 4 | 学科（普通・工業・商業等） |
| 9 | 所在地（市区町村） |

本研究では偏差値・男女別・学科・所在地を取得する。なお設置者については、分析対象を公立高校に限定するためのフィルタとして使用する。

### 2-2. 偏差値データ取得関数の作成

偏差値データを取得する関数を作成する。偏差値の値が正しく取得できているかの確認として、tdタグから取得した文字列が２桁の整数であるかどうかを１文字ずつ検証する。これにより「-」や「非公表」などの文字列を誤って偏差値として取得することを防いでいる。

In [36]:
def get_hensachi(pref_key, pref_ja, delay=2.0):
    url = 'https://xn--swqwd788bm2jy17d.net/' + pref_key + '.php'
    try:
        r = requests.get(url, timeout=15)
        soup = BeautifulSoup(r.text, 'html.parser')
        records = list()

        for table in soup.find_all('table'):
            for tr in table.find_all('tr'):
                tds = tr.find_all('td')
                if len(tds) < 2:
                    continue
                hensachi_text = tds[0].text.strip()
                school_name   = tds[1].text.strip()

                # 偏差値は2桁の整数かどうかを1文字ずつ確認する
                is_number = True
                for char in hensachi_text:
                    if char not in '0123456789':
                        is_number = False
                if not (is_number and len(hensachi_text) == 2):
                    continue

                # 追加情報を取得する（列数が足りない場合は空文字）
                school_type = tds[2].text.strip() if len(tds) > 2 else ''
                gender      = tds[3].text.strip() if len(tds) > 3 else ''
                gakka       = tds[4].text.strip() if len(tds) > 4 else ''
                location    = tds[9].text.strip() if len(tds) > 9 else ''

                records.append({
                    'school_name':   school_name,
                    'hensachi':      int(hensachi_text),
                    'school_type':   school_type,
                    'gender':        gender,
                    'gakka':         gakka,
                    'location':      location,
                    'prefecture':    pref_key,
                    'prefecture_ja': pref_ja,
                })

        time.sleep(delay)
        return records

    except Exception as e:
        print('  エラー：', pref_key)
        time.sleep(delay)
        return list()

In [37]:
# 動作確認（東京都の上位5件）
test_h = get_hensachi('tokyo', '東京都')
for rec in test_h[:5]:
    print(rec['hensachi'], rec['school_name'], rec['school_type'],
          rec['gender'], rec['gakka'], rec['location'])

78 お茶の水女子大学附属高校 国立 女子 普通 文京区
78 開成高校 私立 男子 普通 荒川区
78 筑波大学附属駒場高校 国立 男子 普通 世田谷区
78 筑波大学附属高校 国立 共学 普通 文京区
77 慶應義塾女子高校 私立 女子 普通 港区


### 2-3. 全都道府県の偏差値を収集する

動作確認ができたので、全国全都道府県の偏差値データを取得する。取得後、設置者が都道府県立・市区町村立の公立高校のみに絞る。本研究でkousoku.orgから収集した校則は情報公開請求で開示された公立高校のものであり、私立・国立とは校則文化が根本的に異なるためである。

同一学校に複数の学科がある場合の集約方針は以下のとおりである。偏差値は最大値、男女別は独自ロジック（「女子/商業」のようなスラッシュ表記があれば共学とみなす）、学科は普通科・総合学科が含まれれば「普通」に集約する。

In [38]:
PREF_MAP = [
    ('hokkaido','北海道'), ('aomori','青森県'),   ('iwate','岩手県'),
    ('miyagi','宮城県'),   ('akita','秋田県'),    ('yamagata','山形県'),
    ('fukushima','福島県'),('ibaraki','茨城県'),  ('tochigi','栃木県'),
    ('gunma','群馬県'),    ('saitama','埼玉県'),  ('chiba','千葉県'),
    ('tokyo','東京都'),    ('kanagawa','神奈川県'),('niigata','新潟県'),
    ('toyama','富山県'),   ('ishikawa','石川県'), ('fukui','福井県'),
    ('yamanashi','山梨県'),('nagano','長野県'),   ('gifu','岐阜県'),
    ('shizuoka','静岡県'), ('aichi','愛知県'),    ('mie','三重県'),
    ('shiga','滋賀県'),    ('kyoto','京都府'),    ('osaka','大阪府'),
    ('hyogo','兵庫県'),    ('nara','奈良県'),     ('wakayama','和歌山県'),
    ('tottori','鳥取県'),  ('shimane','島根県'),  ('okayama','岡山県'),
    ('hiroshima','広島県'),('yamaguchi','山口県'),('tokushima','徳島県'),
    ('kagawa','香川県'),   ('ehime','愛媛県'),    ('kouchi','高知県'),
    ('fukuoka','福岡県'),  ('saga','佐賀県'),     ('nagasaki','長崎県'),
    ('kumamoto','熊本県'), ('oita','大分県'),     ('miyazaki','宮崎県'),
    ('kagoshima','鹿児島県'),('okinawa','沖縄県'),
]

# 公立高校の設置者に含まれるキーワード
PUBLIC_KEYWORDS = ['道立','都立','府立','県立','市立','区立','町立','村立','公立']

def is_public(school_type):
    for kw in PUBLIC_KEYWORDS:
        if kw in school_type:
            return True
    return False


def aggregate_gakka(gakka_series):
    """複数学科を1つに集約する。普通科・総合学科が含まれれば「普通」とする。"""
    for g in gakka_series:
        if '普通' in str(g) or '総合' in str(g):
            return '普通'
    return gakka_series.iloc[0]


def aggregate_gender(gender_series):
    """複数行の男女別を1つに集約する。スラッシュ含む場合は共学とみなす。"""
    genders = set()
    for g in gender_series:
        g = str(g)
        if '/' in g:
            return '共学'
        genders.add(g)
    if len(genders) == 1:
        return genders.pop()
    return '共学'

In [39]:
# 既にhensachi.csvが保存済みの場合はTrueに変更する
SKIP_HENSACHI = True

if SKIP_HENSACHI:
    df_hensachi = pd.read_csv('hensachi.csv')
    print(len(df_hensachi), '件の偏差値データを読み込んだ。')

else:
    all_records = list()
    for pref_key, pref_ja in PREF_MAP:
        records = get_hensachi(pref_key, pref_ja)
        if records:
            all_records.extend(records)
            print(pref_ja, '：', len(records), '件')

    df_hensachi_all = pd.DataFrame(all_records)

    # 公立校のみに絞る
    df_hensachi = df_hensachi_all.loc[
        df_hensachi_all['school_type'].apply(is_public), :
    ].copy()

    df_hensachi.to_csv('hensachi.csv', index=False, encoding='utf-8-sig')
    print('収集完了：全', len(df_hensachi_all), '件中、公立', len(df_hensachi), '件を保存した。')

print('偏差値データ合計（公立）：', len(df_hensachi), '件')
print()
print('設置者の種類：')
print(df_hensachi['school_type'].value_counts().head(5))
print()
print('男女別の内訳：')
print(df_hensachi['gender'].value_counts())

6278 件の偏差値データを読み込んだ。
偏差値データ合計（公立）： 6278 件

設置者の種類：
school_type
県立    5436
道立     325
府立     271
都立     246
Name: count, dtype: int64

男女別の内訳：
gender
共学    6219
女子      39
男子      20
Name: count, dtype: int64


In [40]:
df_hensachi.head()

,school_name,hensachi,school_type,gender,gakka,location,prefecture,prefecture_ja
0,札幌西高校,71,道立,共学,普通,札幌市中央区,hokkaido,北海道
1,札幌南高校,71,道立,共学,普通,札幌市中央区,hokkaido,北海道
2,札幌北高校,71,道立,共学,普通,札幌市北区,hokkaido,北海道
3,釧路湖陵高校,68,道立,共学,理数,釧路市,hokkaido,北海道
4,札幌東高校,68,道立,共学,普通,札幌市白石区,hokkaido,北海道


In [ ]:
import requests
import pandas as pd

# =========================
# API設定
# =========================
APP_ID = '3050a2ea7ab556a4f0f801f4d52e9a8db81175fd'

DAYNIGHT_STATS_ID = '0003179188'   # 昼夜間人口比率


# =========================
# データ取得
# =========================
def fetch_values(stats_id, app_id):

    url = 'https://api.e-stat.go.jp/rest/3.0/app/json/getStatsData'

    r = requests.get(
        url,
        params={
            'appId': app_id,
            'statsDataId': stats_id,
            'metaGetFlg': 'N',
            'cntGetFlg': 'N',
            'limit': 100000
        },
        timeout=60
    )

    return r.json()['GET_STATS_DATA']['STATISTICAL_DATA']['DATA_INF']['VALUE']


# =========================
# 地域コード辞書
# =========================
def fetch_area_map(stats_id, app_id):

    url = 'https://api.e-stat.go.jp/rest/3.0/app/json/getMetaInfo'

    r = requests.get(
        url,
        params={
            'appId': app_id,
            'statsDataId': stats_id
        },
        timeout=30
    )

    objs = r.json()['GET_META_INFO']['METADATA_INF']['CLASS_INF']['CLASS_OBJ']

    area_map = {}

    for obj in objs:

        if obj['@id'] == 'area':

            for c in obj['CLASS']:

                if c.get('@level') in ('4', '5', '6'):

                    area_map[c['@code']] = c['@name']

    return area_map


# =========================
# 昼夜間人口比率DataFrame化
# =========================
def build_daynight_df(values):

    rows = []

    for v in values:

        area_code = v.get('@area')

        if area_code is None:
            continue

        try:
            ratio = float(v['$'])
        except:
            continue

        rows.append({
            'area_code': area_code,
            'daynight_ratio': ratio
        })

    return pd.DataFrame(rows)


# =========================
# データ取得
# =========================
values = fetch_values(DAYNIGHT_STATS_ID, APP_ID)


# =========================
# DataFrame化
# =========================
df_daynight = build_daynight_df(values)


# =========================
# 重複除去
# =========================
df_daynight = df_daynight.drop_duplicates(subset='area_code')


# =========================
# 地域名付与
# =========================
area_map = fetch_area_map(DAYNIGHT_STATS_ID, APP_ID)

df_daynight['area_name'] = df_daynight['area_code'].map(area_map)


# =========================
# 市区町村のみ
# =========================
df_daynight = df_daynight[df_daynight['area_name'].notna()]


# 保存
df_daynight.to_csv(
    'df_daynight.csv',
    index=False,
    encoding='utf-8-sig'
)

print(df_daynight.head())


# =========================
# 偏差値データと結合
# =========================
hensachi_daynight = pd.merge(
    df_hensachi,
    df_daynight,
    left_on='location',
    right_on='area_name',
    how='left'
)


# 欠損補完
hensachi_daynight['daynight_ratio'] = (
    hensachi_daynight['daynight_ratio']
    .fillna(100)
)

# area_name削除
hensachi_daynight = hensachi_daynight.drop(
    columns=['area_name']
)

# 保存
hensachi_daynight.to_csv(
    'hensachi_daynight.csv',
    index=False,
    encoding='utf-8-sig'
)

print(hensachi_daynight.head())

  area_code  daynight_ratio area_name
2     01100       1952356.0       札幌市
3     01101        237627.0   札幌市 中央区
4     01102        285321.0    札幌市 北区
5     01103        261912.0    札幌市 東区
6     01104        209584.0   札幌市 白石区
  school_name  hensachi school_type gender gakka location prefecture  \
0       札幌西高校        71          道立     共学    普通   札幌市中央区   hokkaido   
1       札幌南高校        71          道立     共学    普通   札幌市中央区   hokkaido   
2       札幌北高校        71          道立     共学    普通    札幌市北区   hokkaido   
3      釧路湖陵高校        68          道立     共学    理数      釧路市   hokkaido   
4       札幌東高校        68          道立     共学    普通   札幌市白石区   hokkaido   

  prefecture_ja area_code  daynight_ratio  
0           北海道       NaN           100.0  
1           北海道       NaN           100.0  
2           北海道       NaN           100.0  
3           北海道     01206        174742.0  
4           北海道       NaN           100.0  


無事に偏差値データを収集することができた。

### 2-4. DIDデータの取得

#### 2-4-1. DIDとは

DID（Densely Inhabited District：人口集中地区）とは、総務省統計局が国勢調査において設定する地区区分であり、人口密度が$4{,}000\,\text{人}/\text{km}^2$以上の基本単位区が隣接して人口5,000人以上となる地区である。市区町村のDID人口比率は、その地域の都市化度合いを示す指標として広く用いられている。

本研究では、各学校の所在市区町村のDID人口比率を都市度の代理変数として使用する。

#### 2-4-2. e-StatからのDIDデータ取得

DIDデータはe-Stat（政府統計の総合窓口）の国勢調査から取得する。e-StatはAPIを提供しており、市区町村別のDID人口比率を取得できる。なお、e-Stat APIの利用にはアプリケーションIDの取得が必要である。

使用する統計表は「令和2年国勢調査 人口等基本集計 第1-2-1表（統計表ID：0003445079）」で、市区町村別・DID符号別の人口が収録されている。DID人口比率はDID各地区の人口合計を市区町村の総人口で除して算出する。

In [41]:
# APIのアプリケーションID
ESTAT_APP_ID = 'YOUR_APP_ID_HERE'  # 取得したIDに差し替えること

def get_did_ratio_from_estat(app_id):
    """
    e-Stat APIから市区町村別DID人口比率を取得する。
    統計表ID: 0003445079
    令和2年国勢調査 人口等基本集計 第1-2-1表
    """
    r = requests.get(
        'https://api.e-stat.go.jp/rest/3.0/app/json/getStatsData',
        params={
            'appId':       app_id,
            'statsDataId': '0003445079',
            'cdCat02':     '0',        # 男女総数
            'metaGetFlg':  'N',
            'cntGetFlg':   'N',
            'limit':       100000,
        },
        timeout=60
    )
    data = r.json()
    values = data['GET_STATS_DATA']['STATISTICAL_DATA']['DATA_INF']['VALUE']

    print('取得件数：', len(values))

    # 市区町村コード → {総数, DID人口合計} の辞書を作る
    area_data = dict()
    for v in values:
        area_code = v['@area']
        cat01     = v['@cat01']
        try:
            pop = int(v['$'])
        except:
            continue

        if area_code not in area_data:
            area_data[area_code] = {'total': 0, 'did': 0}

        if cat01 == '0000A':
            area_data[area_code]['total'] = pop
        else:
            # DID地区の人口を累積する
            area_data[area_code]['did'] = area_data[area_code]['did'] + pop

    # DID比率を計算する
    records = list()
    for area_code, d in area_data.items():
        if d['total'] > 0:
            did_ratio = round(d['did'] / d['total'] * 100, 2)
        else:
            did_ratio = 0.0
        records.append({
            'area_code': area_code,
            'did_ratio': did_ratio,
        })

    return pd.DataFrame(records)

In [42]:
# 既にdid_ratio.csvが保存済みの場合はTrueに変更する
SKIP_DID = True

if SKIP_DID:
    df_did = pd.read_csv('df_ratio.csv')
    print(len(df_did), '市区町村のDIDデータを読み込んだ。')
else:
    df_did = get_did_ratio_from_estat(ESTAT_APP_ID)
    df_did.to_csv('did_ratio.csv', index=False, encoding='utf-8-sig')
    print('取得完了：', len(df_did), '市区町村')

print()
print('DID比率の基本統計量：')
print(df_did['did_ratio'].describe().round(1))

990 市区町村のDIDデータを読み込んだ。

DID比率の基本統計量：
count    990.0
mean      65.1
std       28.2
min        7.0
25%       40.8
50%       68.0
75%       93.6
max      100.0
Name: did_ratio, dtype: float64


#### 2-4-3. 所在地とDIDデータの名寄せ

メタデータからarea名（市区町村名）→ area_codeの対応表を作成し、高校偏差値.netの所在地（「文京区」「国立市」等）と照合する。

In [44]:
# メタデータからarea名→area_codeの辞書を作成する
r_meta = requests.get(
    'https://api.e-stat.go.jp/rest/3.0/app/json/getMetaInfo',
    params={'appId': ESTAT_APP_ID, 'statsDataId': '0003445079'},
    timeout=30
)
meta = r_meta.json()

area_name_to_code = dict()
for obj in meta['METADATA_INF']['CLASS_INF']['CLASS_OBJ']:
    if obj['@id'] == 'area':
        classes = obj['CLASS']
        if isinstance(classes, list):
            for c in classes:
                level = c.get('@level', '')
                # 市区町村レベル（4・5・6）のみ取得する
                if level in ('4', '5', '6'):
                    area_name_to_code[c['@name']] = c['@code']

print('area名→コード辞書の件数：', len(area_name_to_code))

NameError: name 'df_ratio' is not defined

In [ ]:
# 平成合併後の旧市名を区名として使っているケースを親市名に補正する
# （e-Statには「奥州市水沢区」は登録されておらず「奥州市」として登録されているため）
DISTRICT_TO_CITY = {
    '奥州市水沢区': '奥州市', '奥州市江刺区': '奥州市',
    '奥州市前沢区': '奥州市', '奥州市胆沢区': '奥州市',
    '南相馬市原町区': '南相馬市', '南相馬市小高区': '南相馬市',
    '上越市板倉区': '上越市', '上越市柿崎区': '上越市',
    '上越市安塚区': '上越市', '姫路市大津区': '姫路市',
    '姫路市網干区': '姫路市', '姫路市飾磨区': '姫路市',
}


def location_to_area_code(location):
    """
    hensachi.csvの所在地表記をe-StatのareaCodeに変換する。
    郡名を除去し、合併旧市名は親市名に置き換える。
    """
    loc = str(location).strip()
    # 平成合併旧市名を補正する
    loc = DISTRICT_TO_CITY.get(loc, loc)
    # 「○○郡○○町」の郡名部分を除去する
    loc = re.sub(r'.+郡', '', loc)
    return area_name_to_code.get(loc, None)


df_hensachi['area_code'] = df_hensachi['location'].apply(location_to_area_code)

# DID比率を結合する
did_dict = dict(zip(df_did['area_code'].astype(str), df_did['did_ratio']))
df_hensachi['did_ratio'] = df_hensachi['area_code'].apply(
    lambda x: did_dict.get(str(int(x)), 0.0) if pd.notna(x) else 0.0
)

# マッチ状況の確認
matched = (df_hensachi['area_code'].notna()).sum()
total   = len(df_hensachi)
print('DIDコードマッチ率：', round(matched / total * 100, 1), '%')
print()
print('DID比率の基本統計量（結合後）：')
print(df_hensachi['did_ratio'].describe().round(1))

In [ ]:
# 結合結果のサンプル確認
print('東京の学校（先頭5件）：')
print(df_hensachi.loc[
    df_hensachi['prefecture'] == 'tokyo',
    ['school_name', 'location', 'did_ratio']
].head(5).to_string(index=False))

---

## 3. テキスト前処理

### 3-1. 定型文の特定

スクレイピングで取得した `raw_text` には、校則本文以外の文字列が混在している。何を除去すべきかを判断するため、まず全校のテキストを行単位に分割し、出現校数の多い行を集計することで定型文を特定する。

In [ ]:
# 全校のテキストを行単位に分割し、各行が何校に出現するかを集計する
line_count_dict = dict()

for text in df_schools['raw_text']:
    # 1校につき同じ行を重複してカウントしないようにsetを使う
    seen = set()
    for line in str(text).split('\n'):
        line = line.strip()
        if len(line) == 0:
            continue
        if line not in seen:
            seen.add(line)
            if line in line_count_dict:
                line_count_dict[line] = line_count_dict[line] + 1
            else:
                line_count_dict[line] = 1

# 出現回数の多い順に並び替える
sorted_lines = sorted(line_count_dict.items(), key=lambda x: x[1], reverse=True)

print('出現校数が多い行 上位30件：')
for line, count in sorted_lines[:30]:
    print(str(count).rjust(5), '校  ', repr(line[:80]))

上記の結果から、除去すべき定型文を以下の方針で判断した。

全校（1,663校）に共通して出現する行はサイト共通のUIであり、校則本文とは無関係であるため除去する。それ以外の行については、共通していないということは学校によって内容が異なる可能性があるため、分析の公平性を保つために手を加えない。

全校共通の行を確認したところ、「シェアする」が全校のテキストの末尾に安定して出現することが確認できた。この行以降にはSNSシェアボタン・問題報告フォーム・関連リンクが続いているため、「シェアする」をカットポイントとして、それ以降をまとめて除去する。

### 3-2. テキストのクリーニング

In [ ]:
def clean_text(raw):
    text = str(raw)

    # 「シェアする」以降はサイト共通のUIのため除去する
    if 'シェアする' in text:
        text = text.split('シェアする')[0]

    text = text.replace('\u3000', ' ')  # 全角スペースを半角に変換する
    text = text.replace('\t', ' ')
    return text.strip()

In [ ]:
# 動作確認（末尾が除去されているかを確認する）
sample_raw   = df_schools['raw_text'].iloc[0]
sample_clean = clean_text(sample_raw)

print('=== 処理前（末尾150文字）===')
print(sample_raw[-150:])
print()
print('=== 処理後（末尾150文字）===')
print(sample_clean[-150:])

### 3-3. 文単位への分割

クリーニング済みのテキストを条文単位に分割する。句点（。）の後に改行を挿入してから行単位で分割し、空行のみの行を除外してリスト化する。

In [ ]:
def split_sentences(text):
    # 句点の後に改行を挿入して行ごとに分割する
    text = text.replace('。', '。\n')
    text = text.replace('．', '．\n')
    sentences = list()
    for line in text.split('\n'):
        line = line.strip()
        if line != '':
            sentences.append(line)
    return sentences

In [ ]:
df_schools['clean_text']     = df_schools['raw_text'].apply(clean_text)
df_schools['sentences']      = df_schools['clean_text'].apply(split_sentences)
df_schools['sentence_count'] = df_schools['sentences'].apply(len)
df_schools['char_count']     = df_schools['clean_text'].apply(len)

print('テキスト前処理後の基本統計量：')
print(df_schools[['sentence_count', 'char_count']].describe().round(1))

---

## 4. MeCab形態素解析

校則テキストにどのような語が多く登場するかを把握するため、形態素解析によって名詞・動詞・形容詞を抽出し、頻度を集計する。これにより、収集したデータが校則として適切に取得できているかを確認する。

**unidic-liteの出力フォーマットについて**

使用する辞書 unidic-lite の出力は、一般的なIPAdicと形式が異なる。実際の出力をデバッグして確認した結果、タブ区切りで `[0]表層形 [1]読み [2]正規化読み [3]基本形 [4]品詞-細分類 ...` という形式であることが判明した。この確認を行わずに実装すると、品詞の条件分岐が全て失敗して空リストが返り続けるというバグに遭遇した。

In [ ]:
tagger = MeCab.Tagger()

# unidic-liteの出力フォーマットを実際に確認する
print('=== unidic-lite 出力フォーマット確認 ===')
debug = tagger.parse('制服を着用すること。禁止する。心がける。')
for line in debug.split('\n')[:8]:
    print(repr(line))

上記から、`parts[3]` が基本形（原形）、`parts[4].split('-')[0]` が品詞の大分類であることが確認できた。これをもとに、テキストから内容語を抽出する関数を実装する。

In [ ]:
STOPWORDS = {
    'する', 'ある', 'いる', 'なる', 'もの', 'こと',
    'これ', 'その', 'この', 'よう', 'ため', 'とき',
    '有る', '在る', '居る', '為る', '成る', '成す',
}

def get_content_words(text):
    # unidic-liteのフォーマット（上のデバッグで確認済み）:
    # parts[3] = 基本形、parts[4].split('-')[0] = 品詞の大分類
    result = tagger.parse(text)
    words = list()
    for line in result.split('\n'):
        if line in ('EOS', '') or '\t' not in line:
            continue
        parts = line.split('\t')
        if len(parts) < 5:
            continue
        base = parts[3]
        pos  = parts[4].split('-')[0]
        if pos in ('名詞', '動詞', '形容詞'):
            if len(base) > 1 and base not in STOPWORDS:
                has_digit = False
                for char in base:
                    if char in '0123456789０１２３４５６７８９':
                        has_digit = True
                # ループが終わってから判定する（インデントのミスに注意）
                if not has_digit:
                    words.append(base)
    return words

In [ ]:
# 動作確認
test_sent = '携帯電話の使用を禁止する。やむを得ない場合は担任に届け出ること。'
test_words = get_content_words(test_sent)
print('テスト文：', test_sent)
print('抽出語 ：', test_words)

In [ ]:
# 全校テキストの頻出語を集計する
word_count_dict = dict()
for text in df_schools['clean_text']:
    for word in get_content_words(text):
        if word in word_count_dict:
            word_count_dict[word] = word_count_dict[word] + 1
        else:
            word_count_dict[word] = 1

word_series = pd.Series(word_count_dict)
top30 = word_series.sort_values(ascending=False).head(30)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(top30)), top30.values, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(top30)))
ax.set_xticklabels(top30.index.tolist(), rotation=45, ha='right')
ax.set_title('校則テキストの頻出語 上位30語（全国）')
ax.set_ylabel('出現回数')
plt.tight_layout()
plt.savefig('fig_word_freq.png', dpi=150)
plt.show()

「場合」「生徒」「着用」「禁止」「制服」などが上位に並んでおり、校則として適切なテキストが収集できていることが確認できた。

---

## 5. 自動アノテーション

先行研究（大森, 2023）では条文を手作業で分類した。本研究ではこの手作業を `in` 演算子によるパターンマッチングで自動化する。

### 5-1. アノテーションの設計

各条文にP（禁止）・O（義務）・R（推奨）・E（例外）・A（曖昧）・OTHERの6ラベルのいずれかを付与する。各ラベルは単語の表層形ではなく、その表現が生徒をどのように制約・方向づけるかという観点から定義した。

| ラベル | 定義 | 代表的な表現 |
|-------|-------------|-------------|
| P（禁止） | 生徒の行為を直接制限・禁止する | 「禁止する」「してはならない」「認めない」 |
| O（義務） | 学校が特定の行動を義務として要求する | 「〜すること」「〜しなければならない」 |
| R（推奨） | 努力目標・望ましい行動を示す | 「心がける」「努める」「望ましい」 |
| E（例外） | 条件付きの許可や裁量を認める | 「ただし」「〜してよい」「可とする」 |
| A（曖昧） | 解釈を学校側に委ねる曖昧な規範を示す | 「高校生らしい」「華美でない」「節度ある」 |
| OTHER | 上記のいずれにも該当しない | 理念・前文・手続説明等 |

パターンリストは以下の手順で構築した。まず各ラベルの候補となる表現を列挙し、全ての条文に対して出現件数を集計した。出現頻度が高い表現ほど、校則テキストにおいて安定して使われていると考えられるため、この頻度を根拠としてパターンリストを決定した。

また、辞書は当初P・O・R・Eの4ラベルのみで設計していた。しかし、実際の校則テキストを分析すると、「高校生らしい」「華美でない」など、明示的な禁止ではないものの、解釈を学校に委ねる曖昧なルールが多数存在することが判明した。これらの表現は生徒の行動を直接禁止するのではなく、解釈を通じて間接的に行動を方向づける特徴を持つ（裁量的統治）。このためA（Ambiguous）ラベルを追加し、辞書を更新した。

In [ ]:
# 各ラベルの候補表現の出現件数を集計する（辞書設計の根拠）
candidate_patterns = {
    'P': [
        '禁止', 'してはならない', 'しないこと', '禁ずる', '禁じる', 'してはいけない',
        '慎む', '許可なく', '無断で', '厳禁', '控えること', '認めない', '不可',
        'いけない', '行ってはならない', '持ち込まない',
        '没収', '預かる', '指導の対象', '特別指導',
    ],
    'O': [
        'すること', 'しなければならない', 'でなければならない', '遵守', '着用する',
        '従うこと', 'するものとする', 'を守ること', 'してください', 'するように',
        'しなくてはならない', '提出すること', '届け出ること', '申し出ること',
        '着用すること', '連絡すること', '行うこと', '注意すること', '指定の', '所定の',
        '厳守', '必ず', 'とする', '義務',
    ],
    'R': [
        '心がける', '努める', '望ましい', 'が好ましい', 'に努力', '励む',
        'よう努める', 'を心がけ', '推奨', '基本とする',
        'に心がける', 'するよう心がけ', 'することが大切', '求められる',
    ],
    'E': [
        'ただし', 'やむを得ない', 'この限りではない', '許可を得',
        'してもよい', 'することができる', 'を除く', '場合に限り',
        '任意', '自由とする', 'してよい', '認める', 'でもよい',
        '可とする', '許可される', '場合はこの限り', '承認を得た',
        '差し支えない', 'してもかまわない', 'この限りでない',
        '特例', '例外',
    ],
    'A': [
        '高校生らしい', '高校生として', '生徒として', '華美', '節度', '品位', '奇抜',
        'ふさわしい', '常識', '清潔', '端正', '適切', '生徒らしい', '良識',
        'にふさわしい', 'にふさわしくない', '高校生らしく', 'マナー',
        '節度ある', '相応しい', '相応しくない', '社会通念', '見苦しくない',
    ],
}

all_sentences = list()
for sents in df_schools['sentences']:
    all_sentences.extend(sents)

print('総条文数：', len(all_sentences))
print()

for label, patterns in candidate_patterns.items():
    print('=== ' + label + ' ===')
    pat_count = dict()
    for pat in patterns:
        count = 0
        for sent in all_sentences:
            if pat in sent:
                count = count + 1
        pat_count[pat] = count
    sorted_pats = sorted(pat_count.items(), key=lambda x: x[1], reverse=True)
    for pat, count in sorted_pats:
        print(str(count).rjust(6), '件  ', repr(pat))
    print()

上記の集計結果をもとに、各パターンの出現頻度と実際の条文サンプルを確認しながら最終的なパターンリストを確定した。

なお、`不可` については「不可欠」「不可能」への誤検出リスクがあるため `不可とする`・`は不可` という形で採用した。また `認める`・`任意` については前後の文脈によって意味が変わる場合があるため、`を認める`・`任意とする` のように語末まで指定した。

`没収`・`預かる`・`指導の対象`・`特別指導` は禁止そのものではなく違反後の対応を示す表現であるため、Pラベルから除外した。同様に `とする` はOラベルの候補として件数が多いが、「本規定は〇〇とする」のような定義文に頻出するためOTHERとの競合が大きく、除外した。

### 5-2. アノテーションの実装

集計結果を踏まえ、最終的なパターンリストを以下のように確定した。ラベルの優先順位はOTHER（精神条文を早期に除外）→ P → E → O → A → R → OTHER（未分類）とした。EをOより先に判定する理由は、「やむを得ない場合は担任に届け出ること」のようにEとOの両パターンが同時に出現する文ではEを優先すべきであるためである。またOをAより先に判定する理由は、「服装は端正でなければならない」のように義務文の中に曖昧語が含まれる場合、文全体の性格は義務であるためである。

In [ ]:
# 精神条文・前文など（早期除外）
OTHER_PATS = [
    '目指し', 'の精神に則', 'に貢献', 'の自覚を持', '人間として',
    'を目的とする', 'の向上を図', '本校生徒は', 'を培う',
]

# P：生徒の行為選択を直接制限・否定する表現
P_PATS = [
    '禁止',            'してはならない',  'しないこと',
    'を禁ずる',        'してはいけない',  '慎む',
    '行ってはならない', '許可なく',        '無断で',
    '厳禁',            '控えること',      '認めない',
    '不可とする',      'は不可',          '許可しない',
    'いけない',        'を持ち込まない',  '持ち込み禁止',
]

# O：学校側が特定行動を義務として要求する表現
O_PATS = [
    'すること',          'しなければならない', 'でなければならない',
    'を遵守',           '着用する',           '従うこと',
    'するものとする',    'しなくてはならない',  '届け出ること',
    '申し出ること',      '提出すること',        'を着用すること',
    '連絡すること',      'を行うこと',          '注意すること',
]

# R：義務ではなく努力目標・望ましい状態を示す表現
R_PATS = [
    '心がける',          '努める',             '望ましい',
    'が好ましい',        'に努力',             '励む',
    'よう努める',        'を心がけ',           'に心がける',
    'するよう心がけ',    'することが大切',
]

# E：規則への裁量・逸脱可能性を認める表現
E_PATS = [
    'ただし',            'やむを得ない',       'この限りでない',
    'この限りではない',  '許可を得',           'してもよい',
    'することができる',  'を除く',             '場合に限り',
    '任意とする',        '自由とする',          'してよい',
    'を認める',          'でもよい',            '可とする',
    '許可される',        '場合はこの限り',      '承認を得た',
    '差し支えない',      'してもかまわない',
]

# A：明示的禁止ではないが解釈権を学校側に委ねる曖昧規範表現（裁量的統治）
A_PATS = [
    '高校生らしい',      '高校生として',       '生徒として',
    '華美',              '節度',               '品位',
    'ふさわしい',        '清潔感',             '端正',
    '適切',              '生徒らしい',          'にふさわしい',
    'にふさわしくない',  '高校生らしく',        '節度ある',
    '相応しい',          '相応しくない',        '社会通念',
    '良識',              '見苦しくない',
]


def annotate(sentence):
    # 優先順位：OTHER（早期除外）→ P → E → O → A → R → OTHER（未分類）
    for pat in OTHER_PATS:
        if pat in sentence:
            return 'OTHER'
    for pat in P_PATS:
        if pat in sentence:
            return 'P'
    for pat in E_PATS:
        if pat in sentence:
            return 'E'
    for pat in O_PATS:
        if pat in sentence:
            return 'O'
    for pat in A_PATS:
        if pat in sentence:
            return 'A'
    for pat in R_PATS:
        if pat in sentence:
            return 'R'
    return 'OTHER'

In [ ]:
# 動作確認：各ラベルが期待通りに付与されるかを確認する
test_cases = [
    ('P',     '携帯電話の使用を禁止する。'),
    ('P',     'ジャンバー類の着用は認めない。'),
    ('P',     '許可なく外出してはならない。'),
    ('O',     '制服を着用すること。'),
    ('O',     '服装は質素で端正でなければならない。'),
    ('R',     '礼儀正しく行動するよう心がける。'),
    ('E',     'やむを得ない場合は担任に申し出ること。'),
    ('E',     'ただし、学校行事の場合はこの限りでない。'),
    ('E',     'セーターの着用を認める。'),
    ('A',     '高校生らしい頭髪を心がけること。'),
    ('A',     '華美でない服装とする。'),
    ('OTHER', '本校生徒は豊かな人間性を目指す。'),
]
print('アノテーション動作確認：')
all_pass = True
for expected, sent in test_cases:
    pred = annotate(sent)
    mark = '✓' if pred == expected else '✗'
    if pred != expected:
        all_pass = False
    print(' ', mark, '期待:', expected, ' 予測:', pred, ' ', sent)
print()
print('全テスト通過' if all_pass else '一部ミスあり → パターンを確認すること')

### 5-3. 全校への適用

動作確認ができたので、全校の校則テキストにアノテーションを施す。

In [ ]:
def count_labels(sentences):
    count_dict = {'P': 0, 'O': 0, 'R': 0, 'E': 0, 'A': 0, 'OTHER': 0}
    for s in sentences:
        label = annotate(s)
        count_dict[label] = count_dict[label] + 1
    return pd.Series({
        'P_count':     count_dict['P'],
        'O_count':     count_dict['O'],
        'R_count':     count_dict['R'],
        'E_count':     count_dict['E'],
        'A_count':     count_dict['A'],
        'OTHER_count': count_dict['OTHER'],
    })


label_df   = df_schools['sentences'].apply(count_labels)
df_schools = pd.concat([df_schools, label_df], axis=1)

# スコア計算に使うP+O+R+Eの合計（Aは別途集計）
df_schools['scored_total'] = (
    df_schools['P_count'] + df_schools['O_count'] +
    df_schools['R_count'] + df_schools['E_count']
)

In [ ]:
# ラベル分布の確認と可視化
totals     = df_schools[['P_count', 'O_count', 'R_count', 'E_count', 'A_count', 'OTHER_count']].sum()
total_sum  = totals.sum()
scored_sum = totals[['P_count', 'O_count', 'R_count', 'E_count']].sum()

print('全校のラベル合計：')
print(totals)
print()
print('各ラベルの割合：')
for label in ['P_count', 'O_count', 'R_count', 'E_count', 'A_count', 'OTHER_count']:
    short = label.replace('_count', '')
    print(' ', short + '：', round(totals[label] / total_sum * 100, 1), '%')
print()
print('スコア対象 (P+O+R+E) の割合：', round(scored_sum / total_sum * 100, 1), '%')
print('OTHER（未分類）の割合        ：', round(totals['OTHER_count'] / total_sum * 100, 1), '%')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    ['P\n禁止', 'O\n義務', 'R\n推奨', 'E\n例外', 'A\n曖昧', 'OTHER\n未分類'],
    totals.values,
    color=['#E24B4A', '#EF9F27', '#639922', '#185FA5', '#9B59B6', '#888780'],
    alpha=0.85
)
ax.set_title('アノテーションラベルの分布（全国合計）')
ax.set_ylabel('条文数')
plt.tight_layout()
plt.savefig('fig_label_dist.png', dpi=150)
plt.show()

約77%がOTHER（未分類）に分類された。当初この結果をアノテーション精度の失敗と捉えていたが、校則テキストを繰り返し確認する中で、この「失敗」自体が校則の本質を示している可能性に気づいた。校則には禁止や義務だけでなく、理念・人格形成・生活態度・教育方針といった記述が大量に含まれており、校則は単なる規則の集合ではなく「望ましい生徒像」を構築する教育的テキストとして機能していた。この発見についてはレポート本文の考察節で詳しく論じる。

---

## 6. 自由度スコアの算出

各学校の校則に対して、以下の式で自由度スコアを計算する。

$$\text{自由度スコア} = \frac{R \times 1 + E \times 2}{P \times 2 + O \times 1 + R \times 1 + E \times 2} \times 100$$

本スコアは、校則テキストにおいて許容・裁量を認める表現（R・E）が禁止・義務表現（P・O）に対してどの程度含まれるかを定量化したものである。そのため、EやRが多い学校ほどスコアは高くなり、PやOが多い学校ほどスコアは低くなる。

重みについては、P（禁止）は生徒の行動を直接制限する最も強い拘束表現、E（例外）は生徒の裁量を最も明示的に認める表現として、それぞれ重み2を付与した。O（義務）とR（推奨）は拘束・許容ともにPやEより弱い中間的な表現として重み1とした。なおAラベルはスコア計算に含めず、別途集計する。

ただし、本指標は明文化された規範表現を対象としており、「そもそも規定が存在しない自由」は測定できないという限界を持つ。この点についてはレポート本文の考察節で詳しく論じる。

### 6-1. 閾値の設定

scored_total（P+O+R+Eの合計）が10件未満の学校は、分母が小さすぎてスコアが実態を反映しない可能性がある。たとえばP=0・E=1だけの場合、スコアは100になるが、これは「自由な学校」ではなくアノテーション対象の条文が少なすぎることによるものである。このため該当する学校のスコアを欠損値（NaN）として分析から除外した。

閾値の妥当性を確認するため、閾値を10・25・50に変えて分布の変化を比較した。

In [ ]:
for threshold in [10, 25, 50]:
    scores = df_schools.loc[df_schools['scored_total'] >= threshold, 'freedom_score']
    n = df_schools['scored_total'].ge(threshold).sum()
    # freedom_scoreはまだ計算前のため、ここではscored_totalのみ確認する
    print('閾値', threshold, '：対象校数 =', n)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_schools['scored_total'], bins=30, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(10, color='red', linestyle='--', label='フィルタ閾値：10')
ax.set_title('学校別 scored_total（P+O+R+E の条文数）の分布')
ax.set_xlabel('scored_total')
ax.set_ylabel('学校数')
ax.legend()
plt.tight_layout()
plt.savefig('fig_scored_total.png', dpi=150)
plt.show()

### 6-2. 自由度スコアの算出

In [ ]:
MIN_SCORED = 10

def calc_score(row):
    if row['scored_total'] < MIN_SCORED:
        return np.nan
    w_P, w_O, w_R, w_E = 2, 1, 1, 2
    numer = row['R_count'] * w_R + row['E_count'] * w_E
    denom = (
        row['P_count'] * w_P + row['O_count'] * w_O +
        row['R_count'] * w_R + row['E_count'] * w_E
    )
    if denom == 0:
        return np.nan
    return round(numer / denom * 100, 2)


df_schools['freedom_score'] = df_schools.apply(calc_score, axis=1)

print('自由度スコア算出校数：', df_schools['freedom_score'].notna().sum())
print()
print('自由度スコアの基本統計量：')
print(df_schools['freedom_score'].describe().round(2))

### 6-3. 自由度スコアの分布

In [ ]:
valid_scores = df_schools.loc[df_schools['freedom_score'].notna(), 'freedom_score']
mean_val = valid_scores.mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(valid_scores, bins=25, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(mean_val, color='red', linestyle='--', label='平均：' + str(round(mean_val, 1)))
ax.set_title('自由度スコアの分布（scored_total>=' + str(MIN_SCORED) + '）')
ax.set_xlabel('自由度スコア（0=厳格, 100=自由）')
ax.set_ylabel('学校数')
ax.legend()
plt.tight_layout()
plt.savefig('fig_freedom_hist.png', dpi=150)
plt.show()

In [ ]:
# スコア上位・下位の確認
# スコアが高い学校が「自由な学校」ではなく「例外規定を大量に明記している学校」である
# 可能性をここで確認する（「規定されない自由」のパラドックス）
df_valid = df_schools.loc[df_schools['freedom_score'].notna(), :]

print('自由度スコアが高い学校（上位10校）：')
print(df_valid.nlargest(10, 'freedom_score')[
    ['school_name', 'prefecture', 'freedom_score', 'scored_total', 'P_count', 'E_count']
].to_string(index=False))
print()
print('自由度スコアが低い学校（下位10校）：')
print(df_valid.nsmallest(10, 'freedom_score')[
    ['school_name', 'prefecture', 'freedom_score', 'scored_total', 'P_count', 'E_count']
].to_string(index=False))
print()
print('スコア上位10校の平均scored_total：', round(df_valid.nlargest(10, 'freedom_score')['scored_total'].mean(), 1))
print('スコア下位10校の平均scored_total：', round(df_valid.nsmallest(10, 'freedom_score')['scored_total'].mean(), 1))
print('全体の平均scored_total          ：', round(df_valid['scored_total'].mean(), 1))

---

## 7. 偏差値データとの結合

kousoku.orgと高校偏差値.netでは学校名の表記が異なる場合がある。たとえば「札幌北高等学校（全日・定時）」と「札幌北高校」のような違いである。以下の手順で学校名を正規化（名寄せ）した後、都道府県と学校名の2変数をキーに `pd.merge` でinner joinする。

1. 設置者名（「東京都立」「神奈川県立」等）を `.replace()` で除去
2. 「高等学校」を `.replace()` で「高校」に統一
3. 「（」・「(」以降の課程・学科情報を `.split()` で除去

inner joinを選択した理由は、両方のデータセットに存在する学校のみを分析対象とするためである。また、同一学校に複数の偏差値がある場合（学科別）は `groupby` で最大値を代表値として採用した。kousoku.orgの校則が学科単位ではなく学校単位で記録されているためである。

In [ ]:
SETSUBISHA = [
    '北海道立', '東京都立', '大阪府立', '京都府立',
    '青森県立', '岩手県立', '宮城県立', '秋田県立', '山形県立', '福島県立',
    '茨城県立', '栃木県立', '群馬県立', '埼玉県立', '千葉県立', '神奈川県立',
    '新潟県立', '富山県立', '石川県立', '福井県立', '山梨県立', '長野県立',
    '静岡県立', '愛知県立', '岐阜県立', '三重県立',
    '滋賀県立', '兵庫県立', '奈良県立', '和歌山県立',
    '鳥取県立', '島根県立', '岡山県立', '広島県立', '山口県立',
    '徳島県立', '香川県立', '愛媛県立', '高知県立',
    '福岡県立', '佐賀県立', '長崎県立', '熊本県立', '大分県立',
    '宮崎県立', '鹿児島県立', '沖縄県立',
    '市立', '区立', '町立', '村立',
]

def normalize_name(name):
    name = str(name)
    for s in SETSUBISHA:
        name = name.replace(s, '')
    name = name.replace('高等学校', '高校')
    name = name.replace('高等専門学校', '高専')
    if '（' in name:
        name = name.split('（')[0]
    if '(' in name:
        name = name.split('(')[0]
    return name.strip()

In [ ]:
# 名寄せの動作確認
check_pairs = [
    ('札幌北高等学校(全日・定時)', '札幌北高校'),
    ('東京都立多摩高等学校',       '多摩高校'),
    ('函館中部高等学校(全日・定時)','函館中部高校'),
]
print('名寄せ変換の確認：')
for before, expected in check_pairs:
    result = normalize_name(before)
    mark   = '✓' if result == expected else '✗'
    print(' ', before, '→', result, mark)

In [ ]:
df_schools['name_norm']    = df_schools['school_name'].apply(normalize_name)
df_hensachi['name_norm']  = df_hensachi['school_name'].apply(normalize_name)

# 同一校に複数の学科がある場合は学校単位で集約する
df_h_dedup = (
    df_hensachi
    .groupby(['prefecture', 'name_norm'], as_index=False, observed=True)
    .apply(lambda g: pd.Series({
        'hensachi':      g['hensachi'].max(),
        'gender':        aggregate_gender(g['gender']),
        'gakka':         aggregate_gakka(g['gakka']),
        'did_ratio':     g['did_ratio'].iloc[0],
        'prefecture_ja': g['prefecture_ja'].iloc[0],
    }))
    .reset_index(drop=True)
)

# 都道府県 × 学校名をキーにinner joinで結合する
df_merged = pd.merge(
    df_schools,
    df_h_dedup,
    on=['prefecture', 'name_norm'],
    how='inner'
)

print('校則データ件数  ：', len(df_schools))
print('偏差値データ件数：', len(df_h_dedup))
print('結合後件数      ：', len(df_merged))
print('マッチ率        ：', round(len(df_merged) / len(df_schools) * 100, 1), '%')

In [ ]:
# マッチしなかった学校の先頭20件を確認する
matched_names = set(df_merged['name_norm'])
unmatched = df_schools.loc[~df_schools['name_norm'].isin(matched_names), 'school_name']
print('マッチしなかった学校（先頭20件）：')
for name in unmatched.head(20):
    print(' ', name)

マッチングに失敗した主な原因は、分校・通信制校・定時制校など高校偏差値.netに掲載のない学校の存在であった。

In [ ]:
# 最終分析対象：偏差値・自由度スコア・DID比率の全てが揃っている学校のみ
df_ana = df_merged.loc[
    df_merged['hensachi'].notna()       &
    df_merged['freedom_score'].notna()  &
    df_merged['did_ratio'].notna(),
:].copy()

df_ana['hensachi']  = df_ana['hensachi'].astype(float)
df_ana['did_ratio'] = df_ana['did_ratio'].astype(float)

# ダミー変数の作成
# 男女別：共学を基準カテゴリとする
df_ana['is_girl'] = df_ana['gender'].apply(lambda x: 1 if x == '女子' else 0)
df_ana['is_boy']  = df_ana['gender'].apply(lambda x: 1 if x == '男子' else 0)

# 専門高校ダミー：普通科・総合学科以外を1とする
SENMON_WORDS = ['工業', '商業', '農業', '水産', '家政', '看護', '福祉', '芸術', '体育', '理数']
def check_senmon(gakka):
    for s in SENMON_WORDS:
        if s in str(gakka):
            return 1
    return 0

df_ana['is_senmon'] = df_ana['gakka'].apply(check_senmon)

print('最終分析対象：', len(df_ana), '校')
print('（名寄せ成功 かつ scored_total>=' + str(MIN_SCORED) + ' かつ 偏差値・DID比率あり）')
print()
print('ダミー変数の内訳：')
print('女子校  ：', df_ana['is_girl'].sum(), '校')
print('男子校  ：', df_ana['is_boy'].sum(),  '校')
print('専門高校：', df_ana['is_senmon'].sum(),'校')
print()
print('基本統計量：')
print(df_ana[['hensachi', 'did_ratio', 'freedom_score',
              'is_girl', 'is_boy', 'is_senmon']].describe().round(2))

---

## 8. 収集データの概要

In [ ]:
print('=' * 50)
print('          データ収集・加工の概要')
print('=' * 50)
print('kousoku.org 収集校数          ：', len(df_schools))
print('偏差値データ（公立）           ：', len(df_hensachi))
print('名寄せ後マッチ数               ：', len(df_merged))
print('最終分析対象                   ：', len(df_ana), '校')
print()
print('先行研究（大森, 2023）          ：32校（手動分類）')
print('本研究                         ：' + str(len(df_ana)) + '校（自動アノテーション）')
print('=' * 50)

---

## 9. データ分析・可視化

### 9-1. 相関ヒートマップ

分析の前に、主要変数間の相関関係を `df.corr()` で確認する。どの変数が自由度スコアと関連しているかを俯瞰的に把握することが目的である。

In [ ]:
cols = ['hensachi', 'did_ratio', 'is_girl', 'is_boy', 'is_senmon',
        'freedom_score', 'P_count', 'char_count']
corr = df_ana[cols].corr()

print('相関行列：')
print(corr.round(3))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, ax=ax, square=True
)
ax.set_title('相関ヒートマップ')
plt.tight_layout()
plt.savefig('fig_heatmap.png', dpi=150)
plt.show()

### 9-2. 重回帰分析（自由度スコア）

RQ「偏差値は都市度・学校属性を統制した上でも自由度スコアと有意な関係を持つか」を検証する。以下のモデルで重回帰分析を実施する。

$$\text{自由度スコア} = \beta_0 + \beta_1 \text{偏差値} + \beta_2 \text{DID比率} + \beta_3 \text{女子校} + \beta_4 \text{男子校} + \beta_5 \text{専門高校} + \varepsilon$$

使用パッケージはstatsmodels（OLS）。係数・標準誤差・t値・p値・VIFを出力する。

In [ ]:
X_cols = ['hensachi', 'did_ratio', 'is_girl', 'is_boy', 'is_senmon']
X = df_ana[X_cols].copy()
X = sm.add_constant(X)
y_freedom = df_ana['freedom_score']

model_freedom = sm.OLS(y_freedom, X).fit()
print(model_freedom.summary())

In [ ]:
# 結果の整理（レポート本文用）
print('=== 自由度スコアの重回帰分析結果 ===')
print('サンプル数       ：', int(model_freedom.nobs))
print('決定係数 R²      ：', round(model_freedom.rsquared, 4))
print('自由度調整済み R²：', round(model_freedom.rsquared_adj, 4))
print('F統計量          ：', round(model_freedom.fvalue, 4))
print('F p値            ：', round(model_freedom.f_pvalue, 6))
print()
print('各係数：')
for var in X_cols:
    coef = round(model_freedom.params[var], 4)
    pval = round(model_freedom.pvalues[var], 4)
    sig  = '**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.')
    print(f'  {var:12s}：β = {coef:8.4f}  p = {pval:.4f}  {sig}')

### 9-3. 重回帰分析（禁止条文数）

自由度スコアは複数ラベルを合成した指標であるため、より解釈が明確な禁止条文数を従属変数とした同一モデルでも分析を行う。相関ヒートマップで偏差値との相関が最も強いのはP_countであり、「偏差値が高い学校ほど禁止条文が少ない」という命題を直接検証する。

In [ ]:
y_pcount = df_ana['P_count']

model_pcount = sm.OLS(y_pcount, X).fit()
print(model_pcount.summary())

In [ ]:
print('=== 禁止条文数の重回帰分析結果 ===')
print('サンプル数       ：', int(model_pcount.nobs))
print('決定係数 R²      ：', round(model_pcount.rsquared, 4))
print('自由度調整済み R²：', round(model_pcount.rsquared_adj, 4))
print('F統計量          ：', round(model_pcount.fvalue, 4))
print('F p値            ：', round(model_pcount.f_pvalue, 6))
print()
print('各係数：')
for var in X_cols:
    coef = round(model_pcount.params[var], 4)
    pval = round(model_pcount.pvalues[var], 4)
    sig  = '**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.')
    print(f'  {var:12s}：β = {coef:8.4f}  p = {pval:.4f}  {sig}')

### 9-4. 多重共線性の確認（VIF）

重回帰分析の前提として、説明変数間に強い多重共線性がないことを確認する。VIF（Variance Inflation Factor）が10を超える場合は多重共線性が問題となる可能性がある。

In [ ]:
X_vif = df_ana[X_cols].copy()
X_vif = sm.add_constant(X_vif)

vif_df = pd.DataFrame()
vif_df['variable'] = X_vif.columns
vif_df['VIF'] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

print('VIF（分散拡大係数）：')
print(vif_df.to_string(index=False))
print()
print('（VIF < 10 であれば多重共線性は問題ない）')

### 9-5. 偏差値帯別の分析

`pd.cut` で偏差値を6段階にカテゴリ化し、各グループの平均自由度スコアとラベル構成比を `groupby` + `agg` で集計・可視化する。

In [ ]:
df_ana['hensachi_band'] = pd.cut(
    df_ana['hensachi'],
    bins=[0, 44, 49, 54, 59, 64, 100],
    labels=['44以下', '45-49', '50-54', '55-59', '60-64', '65以上']
)

band_summary = df_ana.groupby('hensachi_band', observed=True).agg(
    n          = ('freedom_score', 'count'),
    score_mean = ('freedom_score', 'mean'),
    score_std  = ('freedom_score', 'std'),
    P_mean     = ('P_count', 'mean'),
    O_mean     = ('O_count', 'mean'),
    R_mean     = ('R_count', 'mean'),
    E_mean     = ('E_count', 'mean'),
    char_mean  = ('char_count', 'mean'),
    did_mean   = ('did_ratio', 'mean'),
).round(2)

print('偏差値帯別の集計：')
print(band_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
band_labels = band_summary.index.tolist()

# 左：偏差値帯別の平均自由度スコア
axes[0].bar(band_labels, band_summary['score_mean'].tolist(),
            color='steelblue', alpha=0.85, edgecolor='white')
axes[0].set_title('偏差値帯別の平均自由度スコア')
axes[0].set_xlabel('偏差値帯')
axes[0].set_ylabel('平均自由度スコア')

# 右：ラベル構成比の積み上げ棒グラフ
ratio     = df_ana.groupby('hensachi_band', observed=True)[
    ['P_count', 'O_count', 'R_count', 'E_count']].mean()
ratio_pct = ratio.div(ratio.sum(axis=1), axis=0) * 100

bottoms = [0.0] * len(ratio_pct)
colors  = ['#E24B4A', '#EF9F27', '#639922', '#185FA5']
lnames  = ['P:禁止', 'O:義務', 'R:推奨', 'E:例外']
for col, color, lname in zip(['P_count', 'O_count', 'R_count', 'E_count'], colors, lnames):
    vals = ratio_pct[col].tolist()
    axes[1].bar(ratio_pct.index.tolist(), vals, bottom=bottoms,
                color=color, alpha=0.85, label=lname)
    bottoms = [bottoms[i] + vals[i] for i in range(len(vals))]

axes[1].set_title('偏差値帯別のラベル構成比（P/O/R/E）')
axes[1].set_xlabel('偏差値帯')
axes[1].set_ylabel('割合 (%)')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.savefig('fig_band_analysis.png', dpi=150)
plt.show()

### 9-6. 散布図（偏差値 × 自由度スコア）

偏差値と自由度スコアの関係を散布図で可視化する。回帰直線は最小二乗法で手計算して描画する。

In [ ]:
x_series = df_ana['hensachi']
y_series = df_ana['freedom_score']
x_mean   = x_series.mean()
y_mean   = y_series.mean()

slope     = ((x_series - x_mean) * (y_series - y_mean)).sum() / ((x_series - x_mean) ** 2).sum()
intercept = y_mean - slope * x_mean

# 偏差値の係数はmodel_freedomから取得する
beta_h  = round(model_freedom.params['hensachi'], 4)
pval_h  = round(model_freedom.pvalues['hensachi'], 4)
sig_h   = '**' if pval_h < 0.01 else ('*' if pval_h < 0.05 else 'n.s.')

x_line = [x_series.min(), x_series.max()]
y_line = [slope * xi + intercept for xi in x_line]

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(x_series.values, y_series.values,
           alpha=0.4, s=40, color='steelblue', edgecolors='white', linewidth=0.5)
ax.plot(x_line, y_line, color='red', linewidth=1.8,
        label='回帰直線（傾き=' + str(round(slope, 3)) + '）')
ax.set_title(
    '偏差値と自由度スコアの関係（単変量）\n'
    + 'β(偏差値) = ' + str(beta_h)
    + ',  p = ' + str(pval_h) + ' ' + sig_h
    + ',  n = ' + str(len(df_ana))
)
ax.set_xlabel('偏差値')
ax.set_ylabel('自由度スコア（0=厳格, 100=自由）')
ax.legend()
plt.tight_layout()
plt.savefig('fig_scatter_main.png', dpi=150)
plt.show()

---

## 10. 分析サマリー

本ノートブックで実施した分析の結果を整理する。以下の数値はレポート本文の各章に記載した数値と対応している。

In [ ]:
print('=' * 60)
print('                    分析サマリー')
print('=' * 60)
print('収集校数（kousoku.org）        ：', len(df_schools), '校')
print('偏差値データ（公立）           ：', len(df_hensachi), '件')
print('名寄せ後マッチ数               ：', len(df_merged), '校  (',
      round(len(df_merged) / len(df_schools) * 100, 1), '%)')
print('最終分析対象                   ：', len(df_ana), '校')
print()
print('--- 変数の記述統計 ---')
print('偏差値   平均±SD：',
      round(df_ana['hensachi'].mean(), 1), '±',
      round(df_ana['hensachi'].std(), 1))
print('DID比率  平均±SD：',
      round(df_ana['did_ratio'].mean(), 1), '±',
      round(df_ana['did_ratio'].std(), 1))
print('自由度スコア 平均±SD：',
      round(df_ana['freedom_score'].mean(), 2), '±',
      round(df_ana['freedom_score'].std(), 2))
print()
print('--- 重回帰：自由度スコア ---')
print('R²      ：', round(model_freedom.rsquared, 4))
print('adj R²  ：', round(model_freedom.rsquared_adj, 4))
for var in X_cols:
    coef = round(model_freedom.params[var], 4)
    pval = round(model_freedom.pvalues[var], 4)
    sig  = '**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.')
    print(f'  {var:12s}：β = {coef:8.4f}  p = {pval:.4f}  {sig}')
print()
print('--- 重回帰：禁止条文数 ---')
print('R²      ：', round(model_pcount.rsquared, 4))
print('adj R²  ：', round(model_pcount.rsquared_adj, 4))
for var in X_cols:
    coef = round(model_pcount.params[var], 4)
    pval = round(model_pcount.pvalues[var], 4)
    sig  = '**' if pval < 0.01 else ('*' if pval < 0.05 else 'n.s.')
    print(f'  {var:12s}：β = {coef:8.4f}  p = {pval:.4f}  {sig}')
print()
print('--- 先行研究（大森, 2023）との比較 ---')
print('先行研究：32校、手動分類、Excel')
print('本研究  ：' + str(len(df_ana)) + '校、MeCab＋自動アノテーション、重回帰分析、Python')
print('=' * 60)